# Business Understanding

### Background
Maritime transport is a major component of global trade, connecting economies through shipping routes, ports and logistics networks. Understanding how trade volumes, shipping capacity, connectivity and port activity interact is important for analysing the performance and evolution of the global maritime system.

### Problem Statement
Maritime trade data is distributed across multiple sources covering different aspects of the shipping system. Analysing these indicators separately limits the understanding of their relationships, emerging patterns and changes during major disruptions. An integrated analytical approach can provide broader insights into global maritime trade and its future trends.

### Objectives
**General Objective:**  
To analyse global maritime trade patterns, relationships and emerging trends using data analytics, statistical analysis and machine learning.

**Specific Objectives:**
1. Examine historical trends in global maritime trade and related indicators.
2. Investigate relationships among trade, shipping capacity, connectivity and port activity.
3. Analyse the effects of major disruptions on maritime trade and shipping activity.
4. Assess maritime resilience through disruption impact, recovery and stability.
5. Identify unusual port activity and group ports by operational patterns.
6. Develop exploratory forecasts of annual seaborne trade by cargo type and geographical region, where data permits.

### Research Questions
1. How have global maritime trade patterns evolved over time?
2. What relationships exist among trade, shipping capacity, connectivity and port activity?
3. How have major disruptions affected maritime trade and shipping activity?
4. How does the maritime system respond and recover from disruptions?
5. What similarities and unusual activity patterns can be identified across ports?
6. What future trends in seaborne trade by cargo type and region can be estimated?

### Scope and Expected Outcomes
The project uses eight datasets covering seaborne trade, container throughput, merchant fleets, connectivity, port calls, port performance, daily port activity and chokepoint transits.

Expected outcomes include insights into maritime trade patterns, relationships, disruption impacts, port activity and exploratory trade forecasts. Forecasts will be evaluated against a naïve baseline. Findings describe associations rather than establish causation, and forecasts are subject to data coverage and limitations.


# Data Understanding

In [5]:
# imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from statsmodels.tsa.arima.model import ARIMA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error,mean_absolute_percentage_error,silhouette_score,davies_bouldin_score,calinski_harabasz_score)
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


In [11]:
# load daily ports
portwatch = pd.read_csv("Datasets/raw/daily_ports_all.csv")
print(portwatch.shape)
portwatch.head()

(5952054, 30)


,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
0,2021-03-11,2021,3,11,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2020-02-05,2020,2,5,port1325,Tsuruga,Japan,JPN,0,0,...,160,160,0,0,0,543,0,543,543,2
2,2021-03-12,2021,3,12,port1325,Tsuruga,Japan,JPN,0,1,...,242,242,0,1754,0,0,0,1754,1754,3
3,2021-03-13,2021,3,13,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,4
4,2021-03-14,2021,3,14,port1325,Tsuruga,Japan,JPN,0,1,...,320,320,0,2670,0,1087,0,3758,3758,5


In [19]:
portwatch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952054 entries, 0 to 5952053
Data columns (total 30 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   date                     object
 1   year                     int64 
 2   month                    int64 
 3   day                      int64 
 4   portid                   object
 5   portname                 object
 6   country                  object
 7   ISO3                     object
 8   portcalls_container      int64 
 9   portcalls_dry_bulk       int64 
 10  portcalls_general_cargo  int64 
 11  portcalls_roro           int64 
 12  portcalls_tanker         int64 
 13  portcalls_cargo          int64 
 14  portcalls                int64 
 15  import_container         int64 
 16  import_dry_bulk          int64 
 17  import_general_cargo     int64 
 18  import_roro              int64 
 19  import_tanker            int64 
 20  import_cargo             int64 
 21  import                   int64 

In [20]:
portwatch.describe()

,year,month,day,portcalls_container,portcalls_dry_bulk,portcalls_general_cargo,portcalls_roro,portcalls_tanker,portcalls_cargo,portcalls,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
count,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,...,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06
mean,2.022441e+03,6.437370e+00,1.568963e+01,6.170749e-01,3.757427e-01,5.177514e-01,1.377770e-01,7.559642e-01,1.648346e+00,2.404310e+00,...,1.121031e+04,1.593580e+04,4.275932e+03,6.295131e+03,4.177303e+02,6.763786e+01,4.420149e+03,1.105650e+04,1.547669e+04,2.976028e+06
std,2.264716e+00,3.412146e+00,8.803135e+00,2.478460e+00,1.379177e+00,1.478039e+00,5.896698e-01,2.852803e+00,4.420998e+00,6.530501e+00,...,5.532706e+04,6.991363e+04,3.671674e+04,4.729657e+04,1.743020e+03,5.592610e+02,2.778433e+04,6.194919e+04,7.134544e+04,1.718210e+06
min,2.019000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,2.020000e+03,3.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.488014e+06
50%,2.022000e+03,6.000000e+00,1.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.976028e+06
75%,2.024000e+03,9.000000e+00,2.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,...,1.116000e+03,3.014000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.180000e+02,2.207000e+03,4.464041e+06
max,2.026000e+03,1.200000e+01,3.100000e+01,7.600000e+01,8.300000e+01,4.300000e+01,1.900000e+01,1.170000e+02,1.760000e+02,2.180000e+02,...,3.406220e+06,3.706936e+06,2.004273e+06,2.641160e+06,8.454700e+04,3.543000e+04,5.437958e+06,2.657613e+06,5.877828e+06,5.952054e+06


In [21]:
## check portwatch mising values
portwatch.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [22]:
portwatch.duplicated().sum()

np.int64(0)

In [13]:
# load us seaborne
seaborne = pd.read_csv('Datasets/raw/US.SeaborneTrade_20261110_095251.csv')
print(seaborne.shape)
seaborne.head()

(333, 77)


,Economy_Label,CargoType_Label,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2000_Metric_tons_MissingValue,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2001_Metric_tons_MissingValue,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,...,2021_Metric_tons_MissingValue,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2022_Metric_tons_MissingValue,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2023_Metric_tons_MissingValue,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote,2024_Metric_tons_MissingValue
0,World,Crude oil loaded,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN
1,World,Other tanker trade loaded,1171172.0,NaN,NaN,1162476.0,NaN,NaN,1196826.0,NaN,...,NaN,2290940.0,NaN,NaN,2288012.0,NaN,NaN,2284711.0,NaN,NaN
2,World,Dry cargo loaded,3361790.0,NaN,NaN,3416510.0,NaN,NaN,3591200.0,NaN,...,NaN,7433784.0,NaN,NaN,7239276.0,NaN,NaN,7726790.0,NaN,NaN
3,World,Total goods loaded,6345730.0,NaN,NaN,6398217.0,NaN,NaN,6576741.0,NaN,...,NaN,11754284.0,NaN,NaN,11598777.0,NaN,NaN,12054069.0,NaN,NaN
4,World,Crude oil discharged,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN


In [23]:
seaborne.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 77 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Economy_Label                        333 non-null    object 
 1   CargoType_Label                      333 non-null    object 
 2   2000_Metric_tons_in_thousands_Value  323 non-null    float64
 3   2000_Metric_tons_Footnote            0 non-null      float64
 4   2000_Metric_tons_MissingValue        9 non-null      object 
 5   2001_Metric_tons_in_thousands_Value  322 non-null    float64
 6   2001_Metric_tons_Footnote            0 non-null      float64
 7   2001_Metric_tons_MissingValue        9 non-null      object 
 8   2002_Metric_tons_in_thousands_Value  324 non-null    float64
 9   2002_Metric_tons_Footnote            0 non-null      float64
 10  2002_Metric_tons_MissingValue        9 non-null      object 
 11  2003_Metric_tons_in_thousands_Va

In [24]:
seaborne.describe()

,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,2003_Metric_tons_in_thousands_Value,2003_Metric_tons_Footnote,2004_Metric_tons_in_thousands_Value,2004_Metric_tons_Footnote,...,2020_Metric_tons_in_thousands_Value,2020_Metric_tons_Footnote,2021_Metric_tons_in_thousands_Value,2021_Metric_tons_Footnote,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote
count,3.230000e+02,0.0,3.220000e+02,0.0,3.240000e+02,0.0,3.220000e+02,0.0,3.230000e+02,0.0,...,3.230000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0
mean,9.871586e+05,NaN,9.974137e+05,NaN,1.019340e+06,NaN,1.080934e+06,NaN,1.166356e+06,NaN,...,1.830849e+06,NaN,1.900331e+06,NaN,1.906340e+06,NaN,1.883756e+06,NaN,1.966969e+06,NaN
std,1.533092e+06,NaN,1.545379e+06,NaN,1.585122e+06,NaN,1.669515e+06,NaN,1.801689e+06,NaN,...,2.829516e+06,NaN,2.939173e+06,NaN,2.938670e+06,NaN,2.901268e+06,NaN,3.051610e+06,NaN
min,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,...,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN
25%,6.204950e+04,NaN,6.161550e+04,NaN,5.827925e+04,NaN,6.325275e+04,NaN,6.363000e+04,NaN,...,1.232500e+05,NaN,1.228542e+05,NaN,1.261062e+05,NaN,1.285468e+05,NaN,1.338748e+05,NaN
50%,4.190540e+05,NaN,4.285210e+05,NaN,4.487830e+05,NaN,4.721545e+05,NaN,5.050890e+05,NaN,...,7.404920e+05,NaN,7.616535e+05,NaN,7.510270e+05,NaN,7.722395e+05,NaN,7.890845e+05,NaN
75%,1.215596e+06,NaN,1.215856e+06,NaN,1.232880e+06,NaN,1.314100e+06,NaN,1.405542e+06,NaN,...,2.205966e+06,NaN,2.326580e+06,NaN,2.314270e+06,NaN,2.308013e+06,NaN,2.404866e+06,NaN
max,1.269146e+07,NaN,1.279643e+07,NaN,1.315348e+07,NaN,1.386194e+07,NaN,1.498393e+07,NaN,...,2.263529e+07,NaN,2.354976e+07,NaN,2.350857e+07,NaN,2.319755e+07,NaN,2.410814e+07,NaN


In [25]:
# missing values in seaborne
seaborne.isnull().sum()

Economy_Label                            0
CargoType_Label                          0
2000_Metric_tons_in_thousands_Value     10
2000_Metric_tons_Footnote              333
2000_Metric_tons_MissingValue          324
                                      ... 
2023_Metric_tons_Footnote              333
2023_Metric_tons_MissingValue          324
2024_Metric_tons_in_thousands_Value      9
2024_Metric_tons_Footnote              333
2024_Metric_tons_MissingValue          324
Length: 77, dtype: int64

In [26]:
seaborne.duplicated().sum()

np.int64(0)

In [15]:
# load merchant fleet
fleet = pd.read_csv('Datasets/raw/US_MerchantFleet.csv')
print(fleet.shape)
fleet.head()

(63896, 23)


,Year,Economy,Economy Label,ShipType,ShipType Label,Dead weight tons in thousands,Dead weight tons in thousands Footnote,Dead weight tons in thousands Missing value,Percentage of total world,Percentage of total world Footnote,...,Percentage of total fleet Missing value,Number of ships,Number of ships Footnote,Number of ships Missing value,Gross Tonnage in thousands,Gross Tonnage in thousands Footnote,Gross Tonnage in thousands Missing value,Average age of vessels (years),Average age of vessels (years) Footnote,Average age of vessels (years) Missing value
0,1980,0000,World,0,Total fleet,672192.465,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
1,1980,0000,World,1,Oil tankers,337945.534,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
2,1980,0000,World,2,Bulk carriers,181880.282,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
3,1980,0000,World,3,General cargo,112840.897,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
4,1980,0000,World,4,Container ships,10290.114,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected


In [27]:
fleet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63896 entries, 0 to 63895
Data columns (total 23 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Year                                          63896 non-null  int64  
 1   Economy                                       63896 non-null  object 
 2   Economy Label                                 63896 non-null  object 
 3   ShipType                                      63896 non-null  int64  
 4   ShipType Label                                63896 non-null  object 
 5   Dead weight tons in thousands                 58814 non-null  float64
 6   Dead weight tons in thousands Footnote        24 non-null     object 
 7   Dead weight tons in thousands Missing value   5082 non-null   object 
 8   Percentage of total world                     58814 non-null  float64
 9   Percentage of total world Footnote            24 non-null    

In [28]:
fleet.describe()

,Year,ShipType,Dead weight tons in thousands,Percentage of total world,Percentage of total fleet,Number of ships,Gross Tonnage in thousands,Average age of vessels (years)
count,63896.000000,63896.000000,5.881400e+04,58814.000000,58814.000000,24600.000000,2.460000e+04,20023.000000
mean,2004.610852,2.457759,2.174818e+04,5.397872,37.797279,1506.377154,2.017613e+04,24.460755
std,13.783127,1.748450,8.731343e+04,12.848273,37.409220,5927.114144,7.814213e+04,11.648492
min,1980.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1993.000000,1.000000,1.753525e+01,0.008471,5.930415,2.000000,2.652000e+00,16.080000
50%,2006.000000,2.000000,4.118255e+02,0.188330,22.924635,35.000000,1.707100e+02,23.290000
75%,2017.000000,4.000000,6.420298e+03,3.591282,66.899630,543.250000,4.086645e+03,30.906500
max,2026.000000,5.000000,2.528788e+06,100.000000,100.000000,115649.000000,1.737514e+06,109.920000


In [29]:
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [30]:
fleet.duplicated().sum()

np.int64(0)

In [14]:
# load lsci
lsci = pd.read_csv('Datasets/raw/US.LSCI_20261121_091634.csv')
print(lsci.shape)
lsci.head()

(15013, 5)


,Economy_Label,Quarter_Label,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote,Index_average_Q1_2023__100_MissingValue
0,Albania,Q4 2006,5.01,NaN,NaN
1,Albania,Q1 2007,5.48,NaN,NaN
2,Albania,Q2 2007,5.48,NaN,NaN
3,Albania,Q3 2007,5.04,NaN,NaN
4,Albania,Q4 2007,5.19,NaN,NaN


In [31]:
lsci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15013 entries, 0 to 15012
Data columns (total 5 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Economy_Label                            15013 non-null  object 
 1   Quarter_Label                            15013 non-null  object 
 2   Index_average_Q1_2023__100_Value         14724 non-null  float64
 3   Index_average_Q1_2023__100_Footnote      0 non-null      float64
 4   Index_average_Q1_2023__100_MissingValue  289 non-null    object 
dtypes: float64(2), object(3)
memory usage: 586.6+ KB


In [32]:
lsci.describe()

,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote
count,14724.000000,0.0
mean,96.932643,NaN
std,133.808579,NaN
min,2.500000,NaN
25%,22.792500,NaN
50%,47.060000,NaN
75%,106.880000,NaN
max,1341.970000,NaN


In [33]:
lsci.isnull().sum()

Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
dtype: int64

In [34]:
lsci.duplicated().sum()

np.int64(0)

In [16]:
# load container throughput
teu = pd.read_csv('Datasets/raw/US_ContPortThroughput.csv')
print(teu.shape)
teu.head()

(3030, 6)


,Year,Economy,Economy Label,TEU (Twenty foot Equivalent Unit),TEU (Twenty foot Equivalent Unit) Footnote,TEU (Twenty foot Equivalent Unit) Missing value
0,2010,0,World,545520944.0,NaN,NaN
1,2010,5100,Africa,23416816.0,NaN,NaN
2,2010,5110,Northern Africa,12024778.0,NaN,NaN
3,2010,5120,Sub-Saharan Africa,11392038.0,NaN,NaN
4,2010,5121,Eastern Africa,2297961.0,NaN,NaN


In [35]:
teu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Year                                             3030 non-null   int64  
 1   Economy                                          3030 non-null   int64  
 2   Economy Label                                    3030 non-null   object 
 3   TEU (Twenty foot Equivalent Unit)                2900 non-null   float64
 4   TEU (Twenty foot Equivalent Unit) Footnote       19 non-null     object 
 5   TEU (Twenty foot Equivalent Unit) Missing value  130 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 142.2+ KB


In [36]:
teu.describe()

,Year,Economy,TEU (Twenty foot Equivalent Unit)
count,3030.000000,3030.000000,2.900000e+03
mean,2017.000000,1431.004950,4.752831e+07
std,4.321207,1819.965223,1.193780e+08
min,2010.000000,0.000000,9.500000e+03
25%,2013.000000,308.000000,4.678590e+05
50%,2017.000000,612.000000,3.420179e+06
75%,2021.000000,1530.000000,2.146032e+07
max,2024.000000,5706.000000,9.203008e+08


In [38]:
teu.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900
dtype: int64

In [39]:
teu.duplicated().sum()

np.int64(0)

In [17]:
# load chokepoints 
choke = pd.read_csv('Datasets/raw/Daily_Chokepoints_Data.csv')
print(choke.shape)
choke.head()

(80528, 21)


,date,year,month,day,portid,portname,n_container,n_dry_bulk,n_general_cargo,n_roro,...,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
0,2019/01/01,2019,1,1,chokepoint1,Suez Canal,15,15,9,4,...,43,61,1018466,618654,112660,21900,1279775,1771682,3051458,1
1,2021/03/11,2021,3,11,chokepoint1,Suez Canal,16,12,3,2,...,33,56,964354,351621,20733,10326,1021116,1347035,2368151,2
2,2019/01/02,2019,1,2,chokepoint1,Suez Canal,24,4,6,6,...,40,50,1534924,227315,8536,37934,518306,1808711,2327017,3
3,2019/01/03,2019,1,3,chokepoint1,Suez Canal,13,14,9,2,...,38,62,555097,977689,39398,10829,868092,1583014,2451107,4
4,2021/03/12,2021,3,12,chokepoint1,Suez Canal,22,15,3,2,...,42,54,1372498,752113,2341,11084,670038,2138038,2808077,5


In [40]:
choke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80528 entries, 0 to 80527
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   date                    80528 non-null  object
 1   year                    80528 non-null  int64 
 2   month                   80528 non-null  int64 
 3   day                     80528 non-null  int64 
 4   portid                  80528 non-null  object
 5   portname                80528 non-null  object
 6   n_container             80528 non-null  int64 
 7   n_dry_bulk              80528 non-null  int64 
 8   n_general_cargo         80528 non-null  int64 
 9   n_roro                  80528 non-null  int64 
 10  n_tanker                80528 non-null  int64 
 11  n_cargo                 80528 non-null  int64 
 12  n_total                 80528 non-null  int64 
 13  capacity_container      80528 non-null  int64 
 14  capacity_dry_bulk       80528 non-null  int64 
 15  ca

In [41]:
choke.describe()

,year,month,day,n_container,n_dry_bulk,n_general_cargo,n_roro,n_tanker,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
count,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,8.052800e+04,8.052800e+04,80528.000000,80528.000000,8.052800e+04,8.052800e+04,8.052800e+04,80528.000000
mean,2022.443672,6.440542,15.688804,14.387666,21.664167,11.469278,2.315766,20.811072,49.836877,70.647949,3.667380e+05,1.097857e+06,38229.182036,5440.079053,5.997501e+05,1.508265e+06,2.108016e+06,40264.500000
std,2.265885,3.413100,8.800194,20.696034,23.221566,15.890087,3.221351,22.203804,51.798241,71.148095,5.214239e+05,1.282904e+06,48834.982214,8670.923652,8.209306e+05,1.581379e+06,2.105823e+06,23246.575576
min,2019.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000
25%,2020.000000,3.000000,8.000000,1.000000,4.000000,1.000000,0.000000,4.000000,12.000000,16.000000,1.158750e+04,1.355515e+05,3470.000000,0.000000,3.459850e+04,2.281238e+05,2.912115e+05,20132.750000
50%,2022.000000,6.000000,16.000000,6.000000,13.000000,4.000000,1.000000,12.000000,32.000000,46.000000,1.007985e+05,5.545225e+05,17303.500000,0.000000,2.563505e+05,8.743895e+05,1.406020e+06,40264.500000
75%,2024.000000,9.000000,23.000000,17.000000,32.000000,17.000000,3.000000,30.000000,66.000000,95.000000,5.417912e+05,1.732328e+06,55880.250000,7942.500000,8.677295e+05,2.327564e+06,3.258355e+06,60396.250000
max,2026.000000,12.000000,31.000000,172.000000,245.000000,149.000000,23.000000,148.000000,352.000000,472.000000,4.016066e+06,1.317163e+07,388902.000000,83176.000000,6.596353e+06,1.434564e+07,1.632013e+07,80528.000000


In [42]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [43]:
choke.duplicated().sum()

np.int64(0)

In [18]:
# load port calls
calls = pd.read_csv('Datasets/raw/US_PortCallsArrivals_S.csv')
print(calls.shape)
calls.head()

(9300, 9)


,Period,Period Label,Economy,Economy Label,CommercialMarket,CommercialMarket Label,Number of port calls,Number of port calls Footnote,Number of port calls Missing value
0,2018S01,S1 2018,1,Individual economies,0,All ships,NaN,NaN,Not applicable
1,2018S01,S1 2018,1,Individual economies,2,Liquid bulk carriers,NaN,NaN,Not applicable
2,2018S01,S1 2018,1,Individual economies,7,Liquefied petroleum gas carriers,NaN,NaN,Not applicable
3,2018S01,S1 2018,1,Individual economies,8,Liquefied natural gas carriers,NaN,NaN,Not applicable
4,2018S01,S1 2018,1,Individual economies,5,Dry bulk carriers,NaN,NaN,Not applicable


In [44]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9300 entries, 0 to 9299
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Period                              9300 non-null   object 
 1   Period Label                        9300 non-null   object 
 2   Economy                             9300 non-null   int64  
 3   Economy Label                       9300 non-null   object 
 4   CommercialMarket                    9300 non-null   int64  
 5   CommercialMarket Label              9300 non-null   object 
 6   Number of port calls                8810 non-null   float64
 7   Number of port calls Footnote       0 non-null      float64
 8   Number of port calls Missing value  490 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 654.0+ KB


In [45]:
calls.describe()

,Economy,CommercialMarket,Number of port calls,Number of port calls Footnote
count,9300.000000,9300.000000,8.810000e+03,0.0
mean,2821.193011,3.962903,6.490234e+04,NaN
std,2112.189066,2.568352,1.991938e+05,NaN
min,0.000000,0.000000,1.100000e+01,NaN
25%,842.000000,2.000000,1.380500e+03,NaN
50%,1640.000000,4.000000,7.185000e+03,NaN
75%,5221.000000,6.000000,4.047450e+04,NaN
max,5900.000000,8.000000,2.461711e+06,NaN


In [46]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [47]:
choke.duplicated().sum()

np.int64(0)

In [48]:
# load cppi 
cppi = pd.read_excel('Datasets/raw/CPPI 2020-2025.xlsx')
print(cppi.shape)
cppi.head()

(426, 20)


,Port,UN/LOCODE,Territory,World Bank Region,Region average CPPI,World Bank Income Group,Income Group average CPPI,Maritime services region,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
0,Aarhus,DKAAR,Denmark,ECA,-8.086595,HI,1.362073,North Europe,-7.718713,73.0,38.0,22.0,60.0,99.0,86.634702,33.0,0.856896,47.61,125.659404,292.0
1,Abidjan,CIABJ,Côte d’Ivoire,SSA,-71.972350,LMI,-8.954771,West Africa,-66.152693,19.0,-153.0,-144.0,-38.0,-51.0,-127.472673,381.0,0.595764,-70.25,-184.695347,407.0
2,Acajutla,SVAQJ,El Salvador,LAC,-14.986492,UMI,1.615451,Caribbean & Central America,-19.239499,0.0,-7.0,-30.0,-89.0,-98.0,-207.620361,393.0,0.479969,-143.38,-271.860723,71.0
3,Adelaide,AUADL,Australia,EAP,30.729639,HI,1.362073,Australasia & Oceania,-11.011090,-20.0,-2.0,-24.0,-39.0,-8.0,-18.214819,297.0,0.833064,-11.43,-24.999638,213.0
4,Aden,YEADE,Yemen,MENAAP,18.958816,LI,-23.905865,Red Sea,63.164190,NaN,-21.0,-13.0,4.0,-24.0,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
cppi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Port                            424 non-null    object 
 1   UN/LOCODE                       419 non-null    object 
 2   Territory                       419 non-null    object 
 3   World Bank Region               419 non-null    object 
 4   Region average CPPI             419 non-null    float64
 5   World Bank Income Group         419 non-null    object 
 6   Income Group average CPPI       419 non-null    float64
 7   Maritime services region        419 non-null    object 
 8   Region's average cppi           419 non-null    float64
 9   CPPI 2020                       326 non-null    float64
 10  CPPI 2021                       352 non-null    float64
 11  CPPI 2022                       340 non-null    float64
 12  CPPI 2023                       391 

In [50]:
cppi.describe()

,Region average CPPI,Income Group average CPPI,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
count,419.000000,419.000000,419.000000,326.000000,352.000000,340.000000,391.000000,403.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,-1.341200,-1.359779,-0.973744,37.710675,5.373509,-2.009647,6.866905,0.012407,-1.288518,200.500000,0.756255,-1.871100,-0.705937,449.247500
std,29.907600,6.021963,36.738435,48.038120,80.082824,78.582603,68.823008,71.335361,64.520553,115.614301,0.136932,38.801558,90.544433,782.571439
min,-71.972350,-25.469787,-98.080938,-108.000000,-665.000000,-675.000000,-519.000000,-721.000000,-302.583415,1.000000,0.369475,-176.720000,-428.446829,24.000000
25%,-14.986492,1.362073,-19.239499,6.000000,-7.000000,-16.000000,-12.500000,-16.500000,-19.752567,100.750000,0.673786,-12.080000,-25.612862,81.750000
50%,-8.086595,1.362073,-7.718713,24.000000,12.000000,7.860000,6.000000,4.000000,1.371539,200.500000,0.785014,1.000000,2.509513,194.000000
75%,30.729639,1.615451,17.910438,68.000000,39.000000,34.000000,39.000000,27.000000,24.013080,300.250000,0.860420,13.542500,33.262400,457.000000
max,34.424249,1.615451,103.168045,170.000000,152.000000,139.000000,152.000000,146.000000,144.638476,400.000000,0.983194,83.550000,209.356951,7385.000000


In [51]:
cppi.isnull().sum()

Port                                2
UN/LOCODE                           7
Territory                           7
World Bank Region                   7
Region average CPPI                 7
World Bank Income Group             7
Income Group average CPPI           7
Maritime services region            7
Region's average cppi               7
CPPI 2020                         100
CPPI 2021                          74
CPPI 2022                          86
CPPI 2023                          35
CPPI 2024                          23
CPPI 2025                          26
Rank 2025                          26
Berth hours in % of port hours     26
Statistical Index                  26
Administrative Index               26
Number of calls in sample          26
dtype: int64

In [52]:
cppi.duplicated().sum()

np.int64(1)

# Data Preparation.

In [54]:
# portwatch 
portwatch_raw= portwatch.copy()

In [55]:
portwatch_raw.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [56]:
# Seaborne
seaborne.columns

Index(['Economy_Label', 'CargoType_Label',
       '2000_Metric_tons_in_thousands_Value', '2000_Metric_tons_Footnote',
       '2000_Metric_tons_MissingValue', '2001_Metric_tons_in_thousands_Value',
       '2001_Metric_tons_Footnote', '2001_Metric_tons_MissingValue',
       '2002_Metric_tons_in_thousands_Value', '2002_Metric_tons_Footnote',
       '2002_Metric_tons_MissingValue', '2003_Metric_tons_in_thousands_Value',
       '2003_Metric_tons_Footnote', '2003_Metric_tons_MissingValue',
       '2004_Metric_tons_in_thousands_Value', '2004_Metric_tons_Footnote',
       '2004_Metric_tons_MissingValue', '2005_Metric_tons_in_thousands_Value',
       '2005_Metric_tons_Footnote', '2005_Metric_tons_MissingValue',
       '2006_Metric_tons_in_thousands_Value', '2006_Metric_tons_Footnote',
       '2006_Metric_tons_MissingValue', '2007_Metric_tons_in_thousands_Value',
       '2007_Metric_tons_Footnote', '2007_Metric_tons_MissingValue',
       '2008_Metric_tons_in_thousands_Value', '2008_Metric_tons_F

In [58]:
# Create a copy of the original dataset
seaborne_clean = seaborne.copy()

# Identify the annual trade-value columns
value_cols = [col for col in seaborne_clean.columns
    if col.endswith("_Metric_tons_in_thousands_Value")
]

# Reshape from wide format to long format
seaborne_clean = seaborne_clean.melt(id_vars=["Economy_Label", "CargoType_Label"],value_vars=value_cols,var_name="Year",value_name="Trade_thousand_tonnes")

# Extract year from column names
seaborne_clean["Year"] = (seaborne_clean["Year"].str.extract(r"(\d{4})").astype(int))

# Rename columns
seaborne_clean = seaborne_clean.rename(columns={"Economy_Label": "Economy","CargoType_Label": "Cargo_type"})

# Arrange columns
seaborne_clean = seaborne_clean[["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]]

# Convert trade values to numeric
seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"],errors="coerce")

print("Standardized columns:")
print(seaborne_clean.columns.tolist())
seaborne_clean.head()

Standardized columns:
['Year', 'Economy', 'Cargo_type', 'Trade_thousand_tonnes']


,Year,Economy,Cargo_type,Trade_thousand_tonnes
0,2000,World,Crude oil loaded,1812768.0
1,2000,World,Other tanker trade loaded,1171172.0
2,2000,World,Dry cargo loaded,3361790.0
3,2000,World,Total goods loaded,6345730.0
4,2000,World,Crude oil discharged,1812768.0


In [59]:
# Clean categorical values
# Preserve official names and meaningful differences between economy groups.
categorical_columns = ["Economy", "Cargo_type"]

for column in categorical_columns:
    seaborne_clean[column] = (seaborne_clean[column].astype("string").str.normalize("NFC").str.replace("\u00a0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA))

In [60]:
# Convert numeric columns and handle non-finite values
seaborne_clean["Year"] = pd.to_numeric(seaborne_clean["Year"], errors="coerce")

seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"], errors="coerce")

seaborne_clean = seaborne_clean.replace([np.inf, -np.inf], np.nan)

print("Missing values before removal:")
seaborne_clean.isna().sum()

Missing values before removal:


Year                       0
Economy                    0
Cargo_type                 0
Trade_thousand_tonnes    241
dtype: int64

In [61]:
# Remove rows containing missing values
# Do not replace missing trade values with zero.

rows_before = len(seaborne_clean)

seaborne_clean = seaborne_clean.dropna(subset=["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]).copy()

print("Rows removed:", rows_before - len(seaborne_clean))

Rows removed: 241


In [62]:
# Validate years and trade values
invalid_rows = (
    ~seaborne_clean["Year"].between(2000, 2024)
    | seaborne_clean["Year"].mod(1).ne(0)
    | seaborne_clean["Trade_thousand_tonnes"].lt(0)
)

print("Invalid rows:", invalid_rows.sum())

seaborne_clean = seaborne_clean.loc[~invalid_rows].copy()

# Zero trade values are valid and are retained.
seaborne_clean["Year"] = seaborne_clean["Year"].astype("int16")

seaborne_clean["Trade_thousand_tonnes"] = (seaborne_clean["Trade_thousand_tonnes"].astype("float64"))

Invalid rows: 0


In [63]:
# Remove duplicate records
key_columns = ["Year", "Economy", "Cargo_type"]

rows_before = len(seaborne_clean)

# Check whether duplicate keys have conflicting trade values
conflicts = (seaborne_clean.groupby(key_columns)["Trade_thousand_tonnes"].nunique())

if conflicts.gt(1).any():
    print("Warning: Duplicate keys have different trade values.")
    print(seaborne_clean[seaborne_clean.duplicated(key_columns, keep=False)].sort_values(key_columns))
else:
    seaborne_clean = seaborne_clean.drop_duplicates(subset=key_columns,keep="first").copy()

    print("Duplicate rows removed:", rows_before - len(seaborne_clean))

Duplicate rows removed: 0


In [65]:
# Detect potential outliers within each economy and cargo type
# A global Z-score would mix economies of different sizes.
# Use group-specific Z-scores for review only.

grouped_values = seaborne_clean.groupby(["Economy", "Cargo_type"])["Trade_thousand_tonnes"]

group_mean = grouped_values.transform("mean")
group_std = grouped_values.transform("std").replace(0, np.nan)

z_scores = (seaborne_clean["Trade_thousand_tonnes"] - group_mean) / group_std

potential_outliers = (seaborne_clean.loc[z_scores.abs().gt(3)].assign(z_score=z_scores.loc[z_scores.abs().gt(3)]).sort_values("z_score",key=lambda values: values.abs(),ascending=False))

print("Potential outliers requiring review:", len(potential_outliers))
print(potential_outliers.to_string(index=False))

Potential outliers requiring review: 6
 Year                                           Economy                Cargo_type  Trade_thousand_tonnes  z_score
 2024 SIDS (Small island developing States) (UN-OHRLLS)          Crude oil loaded                33922.0 3.721186
 2008                           LDCs: Islands and Haiti      Crude oil discharged                   75.0 3.701860
 2024                                   SIDS: Caribbean          Crude oil loaded                32264.0 3.701154
 2023                                        LDCs: Asia      Crude oil discharged                 1948.0 3.130640
 2008                                            Africa Other tanker trade loaded               142454.0 3.113817
 2008                      Developing economies: Africa Other tanker trade loaded               142412.0 3.113583


In [66]:
# Finalize categorical data types and sort
for column in categorical_columns:
    seaborne_clean[column] = seaborne_clean[column].astype("category")

seaborne_clean = (seaborne_clean.sort_values(["Economy", "Cargo_type", "Year"]).reset_index(drop=True))

In [67]:
# Validate the cleaned dataset
key_columns = ["Year", "Economy", "Cargo_type"]

assert seaborne_clean.notna().all().all()
assert not seaborne_clean.duplicated(subset=key_columns).any()
assert seaborne_clean["Year"].between(2000, 2024).all()
assert seaborne_clean["Trade_thousand_tonnes"].ge(0).all()
assert np.isfinite(seaborne_clean["Trade_thousand_tonnes"]).all()

print("Final shape:", seaborne_clean.shape)

print("\nData types:")
print(seaborne_clean.dtypes)

print("\nMissing values:")
print(seaborne_clean.isna().sum())

Final shape: (8084, 4)

Data types:
Year                        int16
Economy                  category
Cargo_type               category
Trade_thousand_tonnes     float64
dtype: object

Missing values:
Year                     0
Economy                  0
Cargo_type               0
Trade_thousand_tonnes    0
dtype: int64


In [68]:
# Save cleaned dataset in processed folder
seaborne_clean.to_csv("Datasets/processed/SeaborneTrade_Cleaned.csv",index=False)

print("Saved: Datasets/processed/SeaborneTrade_Cleaned.csv")

Saved: Datasets/processed/SeaborneTrade_Cleaned.csv


In [69]:
# merchant fleet
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [72]:
print(fleet.shape)                      
print(fleet["Economy"].head(3).tolist()) 
print(fleet["Economy"].dtype)

(63896, 23)
['0000', '0000', '0000']
object


In [74]:
# make copy
fleet_raw = fleet.copy()
fleet_raw.shape

(63896, 23)

In [75]:
# rename columns
MEASURES = {"DWT_k": "Dead weight tons in thousands","Share_world": "Percentage of total world",
            "Share_fleet": "Percentage of total fleet",
            "Ships": "Number of ships",
            "GT_k": "Gross Tonnage in thousands",
            "Age": "Average age of vessels (years)"}

rename = {"Economy": "Code", "Economy Label": "Economy","ShipType": "ShipTypeCode", "ShipType Label": "ShipType"}
for short, long in MEASURES.items():
    rename[long] = short
    rename[long + " Footnote"] = short + "_footnote"
    rename[long + " Missing value"] = short + "_missing"

fleet = fleet_raw.rename(columns=rename)
print(fleet.columns.tolist())
fleet.head(3)

['Year', 'Code', 'Economy', 'ShipTypeCode', 'ShipType', 'DWT_k', 'DWT_k_footnote', 'DWT_k_missing', 'Share_world', 'Share_world_footnote', 'Share_world_missing', 'Share_fleet', 'Share_fleet_footnote', 'Share_fleet_missing', 'Ships', 'Ships_footnote', 'Ships_missing', 'GT_k', 'GT_k_footnote', 'GT_k_missing', 'Age', 'Age_footnote', 'Age_missing']


,Year,Code,Economy,ShipTypeCode,ShipType,DWT_k,DWT_k_footnote,DWT_k_missing,Share_world,Share_world_footnote,...,Share_fleet_missing,Ships,Ships_footnote,Ships_missing,GT_k,GT_k_footnote,GT_k_missing,Age,Age_footnote,Age_missing
0,1980,0000,World,0,Total fleet,672192.465,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
1,1980,0000,World,1,Oil tankers,337945.534,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
2,1980,0000,World,2,Bulk carriers,181880.282,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected


In [76]:
# Extract the Break flags, one per measure plus a general one
for m in MEASURES:
    fleet["break_" + m] = fleet[m + "_footnote"].eq("Break")

fleet["series_break"] = fleet[["break_" + m for m in MEASURES]].any(axis=1)

print(fleet[["break_" + m for m in MEASURES]].sum())
print("any break:", fleet.series_break.sum())
print(fleet[fleet.series_break].groupby("Economy").Year.unique())

break_DWT_k          24
break_Share_world    24
break_Share_fleet    24
break_Ships          12
break_GT_k           12
break_Age            12
dtype: int64
any break: 24
Economy
France                  [2011]
United Kingdom          [1995]
United States     [2002, 2011]
Name: Year, dtype: object


In [77]:
# Keep the three missing-value reasons
for m in MEASURES:
    v, r = fleet[m], fleet[m + "_missing"]
    print(f"{m:12} value present with a reason: {(v.notna() & r.notna()).sum()} | "
          f"value empty with no reason: {(v.isna() & r.isna()).sum()} | "
          f"reasons: {r.value_counts().to_dict()}")

na = fleet[fleet.DWT_k_missing.eq("Not applicable")]
print(na.groupby("Economy").Year.agg(["min", "max", "size"]).head(12))

DWT_k        value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Share_world  value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Share_fleet  value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Ships        value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 34214, 'Not applicable': 5082}
GT_k         value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 34214, 'Not applicable': 5082}
Age          value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 38791, 'Not applicable': 5082}
                                 min   max  size
Economy                                         
Azerbaijan                      1980  1991    72
Belarus                         1980  1991    72
Bosnia an

In [78]:
# Check that the numeric columns are numeric
print(fleet[list(MEASURES)].dtypes)

DWT_k          float64
Share_world    float64
Share_fleet    float64
Ships          float64
GT_k           float64
Age            float64
dtype: object


In [79]:
# Classify every row
def level(code):
    try:
        c = int(code)
    except ValueError:
        return "unallocated"            # '0000u' = World n.e.s.
    if c == 0:
        return "world"
    if c == 1 or c >= 1000:
        return "grouping"
    return "country"

fleet["level"] = fleet["Code"].map(level)
print(fleet["level"].value_counts())

level
country        44809
grouping       18683
world            282
unallocated      122
Name: count, dtype: int64


## EDA

In [ ]:
# Seaborne


# Modelling 

# Evaluation

# Deployment